In [1]:
import pickle
import pandas as pd
from keras.models import load_model


In [2]:
### Load the trained model, scaler pickle,onehot

model = load_model('./model/model.keras')

In [3]:
## load the encoder and scaler

with open('./pkl/onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

In [4]:
with open('./pkl/label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

In [5]:
with open('./pkl/scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [25]:
# Example input data
input_data = [ {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
},
{
    'CreditScore': 900,
    'Geography': 'France',
    'Gender': 'Female',
    'Age': 80,
    'Tenure': 6,
    'Balance': 20000,
    'NumOfProducts': 5,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 90000
}]

In [26]:
type(input_data)

list

In [27]:
input_df = pd.DataFrame(input_data)
print(input_df)

   CreditScore Geography  Gender  Age  Tenure  Balance  NumOfProducts  \
0          600    France    Male   40       3    60000              2   
1          900    France  Female   80       6    20000              5   

   HasCrCard  IsActiveMember  EstimatedSalary  
0          1               1            50000  
1          1               1            90000  


In [28]:
input_df['Gender']  =  label_encoder_gender.transform(input_df['Gender'])
print(input_df)

   CreditScore Geography  Gender  Age  Tenure  Balance  NumOfProducts  \
0          600    France       1   40       3    60000              2   
1          900    France       0   80       6    20000              5   

   HasCrCard  IsActiveMember  EstimatedSalary  
0          1               1            50000  
1          1               1            90000  


In [29]:
import numpy as np

geo_encoder = np.asarray(label_encoder_geo.transform(input_df[['Geography']]))

print(geo_encoder)

[[1. 0. 0.]
 [1. 0. 0.]]


In [30]:
geo_feature_names = label_encoder_geo.get_feature_names_out(['Geography'])
print(geo_feature_names)

['Geography_France' 'Geography_Germany' 'Geography_Spain']


In [31]:
geo_encoder_df = pd.DataFrame(geo_encoder,columns=geo_feature_names)
print(geo_encoder_df)

   Geography_France  Geography_Germany  Geography_Spain
0               1.0                0.0              0.0
1               1.0                0.0              0.0


In [32]:
## Combine one hot encoder columns with the original data
new_input_df = pd.concat([input_df.drop(["Geography"],axis=1),geo_encoder_df],axis=1)
print(new_input_df)

   CreditScore  Gender  Age  Tenure  Balance  NumOfProducts  HasCrCard  \
0          600       1   40       3    60000              2          1   
1          900       0   80       6    20000              5          1   

   IsActiveMember  EstimatedSalary  Geography_France  Geography_Germany  \
0               1            50000               1.0                0.0   
1               1            90000               1.0                0.0   

   Geography_Spain  
0              0.0  
1              0.0  


In [33]:
## Scaling the input data
scaled_input = scaler.transform(new_input_df)
print(scaled_input)
print(type(scaled_input))

[[-0.53598516  0.91324755  0.10479359 -0.69539349 -0.25781119  0.80843615
   0.64920267  0.97481699 -0.87683221  1.00150113 -0.57946723 -0.57638802]
 [ 2.57733415 -1.09499335  3.90769104  0.34567966 -0.89825077  5.98380764
   0.64920267  0.97481699 -0.1813654   1.00150113 -0.57946723 -0.57638802]]
<class 'numpy.ndarray'>


In [34]:
## PRedict churn
predicted_value = model.predict(scaled_input)
print(predicted_value)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
[[0.04244747]
 [0.9999992 ]]


In [35]:
prediction_probability = predicted_value[0][0]

if prediction_probability > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
